# Clasificación de imágenes conjunto de datos Fashion MNIST
## Actividad Práctica

En esta sección se empleará el conjunto de datos Fashion MNIST, una colección de imágenes que representa prendas de vestir, calzado y accesorios en el mundo de la moda. Fashion MNIST es ampliamente reconocido en el campo del aprendizaje automático y la visión por computadora como una alternativa al conjunto de datos Digits MNIST. Al igual que Digits MNIST, el conjunto de datos Fashion MNIST consta de 70.000 imágenes en blanco y negro. De estas, 60.000 están destinadas al entrenamiento de modelos, mientras que las 10.000 restantes se reservan para pruebas y evaluación de rendimiento. Cada imagen tiene una dimensión de 28 x 28 píxeles y se representa en un solo canal de color.

En este laboratorio, se utilizará el conjunto de datos Fashion MNIST con el objetivo de diseñar y entrenar un modelo neuronal convolucional (CNN) para alcanzar una precisión de clasificación superior al 90% en el conjunto de prueba. Te proporcionaremos una función que crea la arquitectura inicial del modelo CNN y una plantilla de compilación. Tu tarea es ajustar esta arquitectura y la configuración de compilación de acuerdo con tus conocimientos. Tienes total libertad para agregar, eliminar o ajustar elementos en la arquitectura y cambiar los hiperparámetros según sea necesario. Además puedes usar las funciones de validación y visualización de resultados para observar el desempeño de tu modelo. Cuando consideres que tu modelo ha superado el umbral del 90% de precisión, podrás enviarlo para su evaluación interna.


In [ ]:
# Celda con la importación de librerías

import torch                     # Importar PyTorch
import torch.nn as nn            # Módulo de redes neuronales de PyTorch
import torch.optim as optim      # Optimizadores de PyTorch

import numpy as np               # Librería para trabajar con datos matriciales

import matplotlib.pyplot as plt  # Librería para gráficas y visualización

# Inicialización de semillas
torch.manual_seed(42)
np.random.seed(42)

# Dispositivo de cómputo (usa GPU si está disponible)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Conjunto de datos Fashion MNIST
from torchvision.datasets import FashionMNIST

In [ ]:
def load_data(dataset_cls, data_dir="./data"):
    """
    Esta función carga al entorno de trabajo los datos de entrenamiento y prueba de
    Fashion MNIST como tensores de PyTorch.
    Parámetros de entrada:
    dataset_cls = Clase de torchvision.datasets con el conjunto de datos Fashion MNIST
    data_dir = Carpeta donde se descargan/almacenan los datos
    Parámetros de salida:
    X_train = Conjunto de datos para entrenamiento, tensor de forma (60000, 1, 28, 28)
    X_test = Conjunto de datos para prueba, tensor de forma (10000, 1, 28, 28)
    y_train = Etiqueta del conjunto de entrenamiento (índices de clase 0-9)
    y_test = Etiqueta del conjunto de prueba (índices de clase 0-9)
    """

    # Descarga (si es necesario) y carga del conjunto de entrenamiento y de prueba
    train_raw = dataset_cls(root=data_dir, train=True, download=True)
    test_raw = dataset_cls(root=data_dir, train=False, download=True)

    # Normalización de los valores de los píxeles de las imágenes
    X_train = train_raw.data.float() / 255.0
    X_test = test_raw.data.float() / 255.0

    # Etiquetas originales (PyTorch usa índices de clase, no one-hot, junto con
    # nn.CrossEntropyLoss)
    y_train = train_raw.targets
    y_test = test_raw.targets

    print(f'Conjunto de datos cargados')
    print(f'Tamaño imágenes de entrenamiento {tuple(X_train.shape)}')
    print(f'Tamaño imágenes de test {tuple(X_test.shape)}')
    print(f'Datos por cada etiqueta en X train {np.bincount(y_train.numpy())}')

    # Se agrega la dimensión del canal (PyTorch espera (N, C, H, W))
    return X_train.unsqueeze(1), X_test.unsqueeze(1), y_train, y_test

In [ ]:
X_train, X_test, y_train, y_test = load_data(FashionMNIST)

In [ ]:
def visualize_instances(X_data, y_data):
    """
    Esta función visualiza una instancia de cada categoría del conjunto Fashion MNIST
    Parámetros de entrada:
    X_data = Instancias, tensor de forma (N, 1, 28, 28)
    y_data = Etiquetas
    Parámetros de salida:
    Gráfica con la visualización de la imagen por cada instancia.
    """
    fig, ax = plt.subplots(nrows=2, ncols=5, sharex=True, sharey=True, figsize=(10, 5))
    ax = ax.flatten()
    for i in range(10):
        img = X_data[y_data == i][0, 0]
        ax[i].imshow(img, cmap='gist_yarg')
        ax[0].set_xticks([])
        ax[0].set_yticks([])
    return plt.show()

In [ ]:
visualize_instances(X_train, y_train)

In [ ]:
def load_model():
    """
    Esta función construye un modelo neuronal convolucional para el problema de
    clasificación con Fashion MNIST.

    Modifica esta función a tu gusto.

    Salida:
    model = modelo neuronal convolucional en PyTorch (nn.Module)
    """

    # Modifica las capas como consideres necesario

    # Ingresa tu código aqui

    # Nota: no se agrega una activación softmax en la capa de salida porque
    # nn.CrossEntropyLoss la aplica internamente (junto con la funcion log)

    model = nn.Sequential(
        # Primera capa convolucional. Kernel de 3x3 y 2 filtros
        nn.Conv2d(in_channels=1, out_channels=2, kernel_size=3, padding=1),
        nn.ReLU(),
        # Capa de MaxPooling
        nn.MaxPool2d(kernel_size=2),
        # Capa de aplanamiento
        nn.Flatten(),
        # Capa de salida
        nn.Linear(2 * 14 * 14, 10)
    )

    return model

In [ ]:
def evaluate_model(model, X, y, batch_size=128, criterion=None):
    """
    Evalúa `model` sobre (X, y) y calcula la pérdida y precisión promedio,
    de forma equivalente a model.evaluate() en tensorflow.keras.
    Parámetros de entrada:
    model = modelo de PyTorch (nn.Module)
    X, y = datos y etiquetas sobre los que evaluar
    batch_size = tamaño de lote usado para recorrer los datos
    criterion = función de pérdida (por defecto nn.CrossEntropyLoss)
    Salida:
    loss = pérdida promedio
    accuracy = precisión promedio
    """
    if criterion is None:
        criterion = nn.CrossEntropyLoss()

    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            X_batch = X[i:i + batch_size].to(device)
            y_batch = y[i:i + batch_size].to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            total_loss += loss.item() * len(X_batch)
            correct += (outputs.argmax(dim=1) == y_batch).sum().item()
            total += len(X_batch)

    return total_loss / total, correct / total


def train_model(model, X_train, y_train, X_test, y_test, optimizer,
                 epochs=10, batch_size=32, criterion=None):
    """
    Entrena `model` durante `epochs` épocas usando mini-lotes de tamaño `batch_size`,
    evaluando en cada época sobre (X_test, y_test), de forma equivalente a
    model.fit(..., validation_data=(X_test, y_test)) en tensorflow.keras.
    Parámetros de entrada:
    model = modelo de PyTorch (nn.Module)
    X_train, y_train = datos y etiquetas de entrenamiento
    X_test, y_test = datos y etiquetas de validación/prueba
    optimizer = optimizador de torch.optim ya asociado a los parámetros de `model`
    epochs = número de épocas de entrenamiento
    batch_size = tamaño de mini-lote
    criterion = función de pérdida (por defecto nn.CrossEntropyLoss)
    Salida:
    history = diccionario con las llaves 'accuracy', 'val_accuracy', 'loss', 'val_loss'
    """
    if criterion is None:
        criterion = nn.CrossEntropyLoss()

    model.to(device)
    n_train = len(X_train)
    history = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n_train)
        running_loss, correct, total = 0.0, 0, 0

        for i in range(0, n_train, batch_size):
            idx = perm[i:i + batch_size]
            X_batch = X_train[idx].to(device)
            y_batch = y_train[idx].to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(X_batch)
            correct += (outputs.argmax(dim=1) == y_batch).sum().item()
            total += len(X_batch)

        train_loss, train_acc = running_loss / total, correct / total
        val_loss, val_acc = evaluate_model(model, X_test, y_test, batch_size=batch_size, criterion=criterion)

        history['accuracy'].append(train_acc)
        history['val_accuracy'].append(val_acc)
        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        print(f"Epoch {epoch + 1}/{epochs} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} "
              f"- val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

    return history


In [ ]:
# Compilar y entrenar el modelo

# También puedes modificar esta celda, cambiando parámetros de la compilación o del
# entrenamiento.

model = load_model()
optimizer = optim.SGD(model.parameters(), lr=0.01)
resultsFit = train_model(model, X_train, y_train, X_test, y_test,
                          optimizer=optimizer,
                          epochs=10,
                          batch_size=32)

In [ ]:
def visualization_history(resultsFit):
    """
    Esta función grafica las curvas de precisión a lo largo de las épocas de entrenamiento
    para el conjunto de prueba como el de validación.
    Parámetros:
    resultsFit = historial de entrenamiento (diccionario con las llaves 'accuracy',
    'val_accuracy', 'loss', 'val_loss')
    Salida:
    Gráfica de la precisión en validación y prueba.
    """
    # Llamado al entrono de la gráfica y fijar tamaño de visualización
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Graficar la presición de la historia en entrenamiento
    axes[0].plot(range(len(resultsFit['accuracy'])), resultsFit['accuracy'],
           '#5a2ff4', linestyle = "-", marker = "o",
           label='Precisión entrenamiento', lw = 3)

    # Graficar la presición de la historia en validación
    axes[0].plot(range(len(resultsFit['val_accuracy'])), resultsFit['val_accuracy'],  '#60d13e',
           label='Precisión validación', linestyle = '-', marker = 'o', lw = 3)

    # Umbral 90% de precisión
    axes[0].plot(range(len(resultsFit['val_accuracy'])), 0.9*np.ones(len(resultsFit['val_accuracy'])),
               'red', lw = 3, label = 'Umbral')

    # Títulos de los ejes
    axes[0].set_xlabel('Época',fontsize=16)
    axes[0].set_ylabel(r'Precisión',fontsize=16)
    axes[0].set_ylim(top=1) # Límite de la grafica
    axes[0].grid()      # Activar la cuadricula
    axes[0].legend(fontsize=14,loc = 'lower right')   # Establecer la legenda


    # Graficar la pérdida de la historia en entrenamiento
    axes[1].plot(range(len(resultsFit['loss'])), resultsFit['loss'],
           '#5a2ff4', linestyle = "-", marker = "o",
           label='Pérdida entrenamiento', lw = 3)

    # Graficar la presición de la historia en validación
    axes[1].plot(range(len(resultsFit['val_loss'])), resultsFit['val_loss'],  '#60d13e',
           label='Pérdida validación', linestyle = '-', marker = 'o', lw = 3)

    # Títulos de los ejes
    axes[1].set_xlabel('Época',fontsize=16)
    axes[1].set_ylabel(r'Pérdida',fontsize=16)
    axes[1].grid()      # Activar la cuadricula
    axes[1].legend(fontsize=14,loc = 'upper right')   # Establecer la legenda


    plt.tight_layout() # Ajusta automáticamente el diseño de la figura
    plt.show()
### END READ ONLY

In [ ]:
# Llamado a la función de visualización de las métricas de la historia del entrenamiento
# del modelo, en la prrimera gráfica se presenta la precisión y en la segunda la pérdida
visualization_history(resultsFit)

**Verifica que tu modelo supere el umbral de precisión del 90%, para esto las curvas de precisión en el conjunto de prueba, (curva de color verde) debe situarse por encima de la linea de color rojo.** Cuando estes listo selecciona el botón **Enviar tarea**, al enviar tu laboratorio se calificará automáticamente.

**Ayuda:** Puedes intentar cambiando el optimizador a la clase **adam**, modificar en la arquitectura base de 2 filtros a 16 en la primera capa convolucional y agregar otra capa convolucional de 32 filtros. Probando esta configuración de seguro se supera el umbral

In [ ]:
_, check = evaluate_model(model, X_test, y_test)
check